# Task 2: Metadata Schema & Chunking

**Goal:** Implement chunking with metadata schema per WEEK3_TASKS.md §3.2

**Decisions:**
- Chunk-page: Option B (multiple chunks per page, never cross boundaries)
- Metadata: Add `chunk_index` and `char_count` beyond §3.2 spec
- Phase 2 placeholders: `section`, `step_number`, `image_ids`

## 1. Setup

In [2]:
import re
from pathlib import Path
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_DIR = Path("../data/raw_pdfs")
TEST_PDF = PDF_DIR / "waterpurifier_complex.pdf"

print(f"PDF exists: {TEST_PDF.exists()}")

PDF exists: True


## 2. Filename Parser

Extract `category`, `complexity`, `model_name` from filename.

Naming convention: `{category}_{complexity}*.pdf`

In [3]:
def parse_filename(filepath: Path) -> dict:
    """Extract metadata from PDF filename.
    
    Expected format: {category}_{complexity}*.pdf
    Examples:
        - waterpurifier_complex.pdf
        - airpurifier_simple.pdf
        - vacuumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf
    """
    filename = filepath.stem  # without extension
    
    # Extract category (first part before _)
    # Valid categories: waterpurifier, airpurifier, vacuumcleaner
    category_match = re.match(r'^(waterpurifier|airpurifier|vacuumcleaner)', filename, re.IGNORECASE)
    category = category_match.group(1).lower() if category_match else "unknown"
    
    # Extract complexity (simple or complex after category_)
    complexity_match = re.search(r'_(simple|complex)', filename, re.IGNORECASE)
    complexity = complexity_match.group(1).lower() if complexity_match else "unknown"
    
    # Extract model_name - try to find model pattern (e.g., WD520AWB, AS181DAW)
    # LG model pattern: 2-3 letters + numbers + optional letters
    model_match = re.search(r'([A-Z]{2,3}\d{3,}[A-Z]*)', filename, re.IGNORECASE)
    if model_match:
        model_name = model_match.group(1).upper()
    else:
        # Fallback: use category_complexity as identifier
        model_name = f"{category}_{complexity}"
    
    return {
        "source": filepath.name,
        "category": category,
        "complexity": complexity,
        "model_name": model_name,
    }

# Test on all PDFs
for pdf in sorted(PDF_DIR.glob("*.pdf")):
    meta = parse_filename(pdf)
    print(f"{pdf.name}")
    print(f"  -> {meta}")
    print()

airpurifier_complex_MFL69726859_00_190321_00.pdf
  -> {'source': 'airpurifier_complex_MFL69726859_00_190321_00.pdf', 'category': 'airpurifier', 'complexity': 'complex', 'model_name': 'MFL69726859'}

airpurifier_simple.pdf
  -> {'source': 'airpurifier_simple.pdf', 'category': 'airpurifier', 'complexity': 'simple', 'model_name': 'airpurifier_simple'}

vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf
  -> {'source': 'vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf', 'category': 'unknown', 'complexity': 'simple', 'model_name': 'MFL68700206'}

vacuumcleaner_complex.pdf
  -> {'source': 'vacuumcleaner_complex.pdf', 'category': 'vacuumcleaner', 'complexity': 'complex', 'model_name': 'vacuumcleaner_complex'}

waterpurifier_complex.pdf
  -> {'source': 'waterpurifier_complex.pdf', 'category': 'waterpurifier', 'complexity': 'complex', 'model_name': 'waterpurifier_complex'}

waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf
  -> {'source': 'waterpurifier_simpl

## 3. Load PDF with PDFPlumberLoader

In [4]:
loader = PDFPlumberLoader(str(TEST_PDF))
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print(f"\nPage 9 sample ({len(pages[8].page_content)} chars):")
print(pages[8].page_content[:500])

Loaded 40 pages

Page 9 sample (1382 chars):
LG ThinQ 사용하기 9
LG ThinQ 사용하기
LG ThinQ와 LG 가전 연결하기 *1 구형 무선 공유기는 유니코드(전 세계의 모든
문자를 다루도록 설계된 표준 문자 전산 처리
방식) UTF-8을 지원하지 않는 공유기입니다.
앱 설치 및 제품 등록하기
LG ThinQ 앱을 설치하면 언제 어디서나 편리하게 우리집 • 정수기와 무선 공유기 사이의 거리가 너무 멀면 신호
LG 가전을 관리할 수 있습니다. 강도가 약해집니다. 신호가 약하면 제품을 등록하는 데
많은 시간이 걸리거나 실패할 수 있습니다.
• 본 내용은 제품에 f 아이콘이 있는 모델에만
• 아래 경우에는 LG ThinQ 앱의 제품별 설정에서
적용됩니다.
네트워크 정보를 변경하세요.
스마트폰의 카메라 또는 QR 코드 리더 앱을 사용하여
- 무선 공유기를 변경했을 때
제품에 부착된 QR 코드를 스캔하세요.
- 무선 공유기의 비밀번호를 변경했을 때
- 인터넷 서비스 제공 업체를 변경했을 때
• 본 설명서의 내용은 LG Thi


## 4. Chunking Strategy

**Option B:** Multiple chunks per page, never cross page boundaries.

- `chunk_size=1000`
- `chunk_overlap=200`
- Process each page separately

In [5]:
def chunk_pdf_with_metadata(
    filepath: Path,
    chunk_size: int = 1000,
    chunk_overlap: int = 200,
) -> list[dict]:
    """Load PDF and split into chunks with metadata.
    
    Chunks never cross page boundaries (Option B).
    """
    # Parse filename for base metadata
    base_meta = parse_filename(filepath)
    
    # Load PDF
    loader = PDFPlumberLoader(str(filepath))
    pages = loader.load()
    
    # Initialize splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    
    all_chunks = []
    
    for page_doc in pages:
        page_num = page_doc.metadata.get("page", 0) + 1  # 1-indexed
        page_text = page_doc.page_content
        
        # Skip empty pages
        if not page_text.strip():
            continue
        
        # Split this page's text
        page_chunks = splitter.split_text(page_text)
        
        for chunk_idx, chunk_text in enumerate(page_chunks):
            chunk_id = f"{base_meta['model_name']}_p{page_num:03d}_c{chunk_idx:03d}"
            
            chunk_data = {
                # Content
                "text": chunk_text,
                # Base metadata from filename
                **base_meta,
                # Page/chunk metadata
                "page": page_num,
                "chunk_id": chunk_id,
                "chunk_index": chunk_idx,
                "char_count": len(chunk_text),
                # Phase 2 placeholders
                "section": None,
                "step_number": None,
                "image_ids": [],
            }
            all_chunks.append(chunk_data)
    
    return all_chunks

In [6]:
# Test on waterpurifier_complex.pdf
chunks = chunk_pdf_with_metadata(TEST_PDF)

print(f"Total chunks: {len(chunks)}")
print(f"\nChunk size distribution:")
sizes = [c["char_count"] for c in chunks]
print(f"  Min: {min(sizes)}, Max: {max(sizes)}, Avg: {sum(sizes)/len(sizes):.0f}")

Total chunks: 56

Chunk size distribution:
  Min: 50, Max: 999, Avg: 672


In [7]:
# Inspect sample chunks
print("=" * 60)
print("Sample chunks from page 9-12:")
print("=" * 60)

for chunk in chunks:
    if chunk["page"] in [9, 10, 11, 12]:
        print(f"\n--- {chunk['chunk_id']} ({chunk['char_count']} chars) ---")
        print(f"Page: {chunk['page']}, Index: {chunk['chunk_index']}")
        print(f"Text preview: {chunk['text'][:200]}...")

Sample chunks from page 9-12:

--- waterpurifier_complex_p009_c000 (998 chars) ---
Page: 9, Index: 0
Text preview: LG ThinQ 사용하기 9
LG ThinQ 사용하기
LG ThinQ와 LG 가전 연결하기 *1 구형 무선 공유기는 유니코드(전 세계의 모든
문자를 다루도록 설계된 표준 문자 전산 처리
방식) UTF-8을 지원하지 않는 공유기입니다.
앱 설치 및 제품 등록하기
LG ThinQ 앱을 설치하면 언제 어디서나 편리하게 우리집 • 정수기와 무선 공유기 사이의 거리...

--- waterpurifier_complex_p009_c001 (556 chars) ---
Page: 9, Index: 1
Text preview: 무선 사양
해당 무선설비는 전파혼신 가능성이 있으므로 인명 안전과
관련된 서비스는 할 수 없습니다.
• 무선 공유기의 인증 및 암호화 유형은 WPA2를 무선 통신
사용 주파수 최대 출력
권장합니다. 방식
• 무선 네트워크 연결 품질은 주변의 무선 환경에 영향을 와이파이 < 12 mW/MHz
2400 MHz ~
받을 수 있습니다. 장애가 발생하면 인터넷 서비...

--- waterpurifier_complex_p010_c000 (984 chars) ---
Page: 10, Index: 0
Text preview: 10 LG ThinQ 사용하기
• 사용 가능한 기능은 구매하신 제품과 LG ThinQ 앱의 • 설정한 시간 동안에는 제품 소리가 무음으로 변경되고
버전에 따라 달라질 수 있습니다. 밝기도 조절됩니다. 밝기는 2단계 (꺼짐, 30%)로 조절할
수 있습니다.
정수기 사용 현황
오늘의 물 사용량 등을 확인할 수 있습니다.
자동 업다운 기능
자동 업다운 기능이 있...

--- waterpurifier_complex_p010_c001 (717 chars) ---
Page: 10, Index: 1
Text preview: • 용량 조절 버튼a이 있는 

## 5. Verify Metadata Schema

In [8]:
# Check that all required fields exist
required_fields = [
    "text", "source", "category", "complexity", "model_name",
    "page", "chunk_id", "chunk_index", "char_count",
    "section", "step_number", "image_ids"
]

sample_chunk = chunks[0]
print("Metadata schema verification:")
print("-" * 40)
for field in required_fields:
    value = sample_chunk.get(field, "MISSING")
    if field == "text":
        value = f"{value[:50]}..." if len(str(value)) > 50 else value
    status = "✅" if field in sample_chunk else "❌"
    print(f"{status} {field}: {value}")

Metadata schema verification:
----------------------------------------
✅ text: 제품 사용설명서
데 스 크 정 수 기
제품을 안전하고 편리하게 사용하기 위해 반드시 제품을...
✅ source: waterpurifier_complex.pdf
✅ category: waterpurifier
✅ complexity: complex
✅ model_name: waterpurifier_complex
✅ page: 1
✅ chunk_id: waterpurifier_complex_p001_c000
✅ chunk_index: 0
✅ char_count: 511
✅ section: None
✅ step_number: None
✅ image_ids: []


## 6. Test on All PDFs

In [ ]:
# Process all 6 PDFs and summarize
all_pdfs = sorted(PDF_DIR.glob("*.pdf"))

print(f"{'PDF':<50} {'Pages':<8} {'Chunks':<8} {'Avg Size':<10}")
print("-" * 76)

total_chunks = 0
for pdf in all_pdfs:
    loader = PDFPlumberLoader(str(pdf))
    pages = loader.load()
    chunks = chunk_pdf_with_metadata(pdf)
    
    avg_size = sum(c["char_count"] for c in chunks) / len(chunks) if chunks else 0
    total_chunks += len(chunks)
    
    print(f"{pdf.name:<50} {len(pages):<8} {len(chunks):<8} {avg_size:<10.0f}")

print("-" * 76)
print(f"{'TOTAL':<50} {'':<8} {total_chunks:<8}")

PDF                                                Pages    Chunks   Avg Size  
----------------------------------------------------------------------------
airpurifier_complex_MFL69726859_00_190321_00.pdf   56       67       538       
airpurifier_simple.pdf                             48       58       603       
vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf 24       26       572       
vacuumcleaner_complex.pdf                          52       69       626       
waterpurifier_complex.pdf                          40       56       672       
waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf 32       44       666       
----------------------------------------------------------------------------
TOTAL                                                       320     


## 7. Notes

### Results Summary
- **Total chunks:** 320 across 6 PDFs
- **Avg chunk size:** 538-672 chars (below 1000 target, expected for short pages)
- **Metadata schema:** All fields present ✅

### Known Limitations

**LIM-003: Model Name Extraction (MINOR)**
- Filename parser cannot reliably extract actual LG model numbers (e.g., WD520AWB)
- Current behavior: Falls back to `{category}_{complexity}` or picks up MFL document IDs
- Impact: chunk_id and model_name fields are inconsistent across PDFs
- Workaround for baseline: **Manual metadata description required** (hardcode model mapping or add to filename)
- Future: Parse model name from PDF first page content

### Future improvements (Advanced RAG)
- Option C (cross page boundaries) may improve semantic continuity
- Consider semantic chunking based on section headers
- Add `summary` field for hierarchical retrieval